# SerpentGuard AI — Combine Sources & Re-clean

Merges extended sources with iNaturalist base. Cross-source deduplication. Append new photos to train only.

In [ ]:
import numpy as np, pandas as pd, imagehash
from pathlib import Path
from PIL import Image
import kagglehub

splits_path = Path(kagglehub.dataset_download("puneetbajajiitm/serpentguard-m2-splits"))
df_base = pd.read_csv(splits_path / "splits.csv")
classes_df = pd.read_csv(splits_path / "classes.csv")
print(f"Base dataset: {len(df_base):,} rows")

# Load extended manifest (from notebook 08 output)
extended_path = Path("/kaggle/input/serpentguard-extended-sources")  # if uploaded
if (extended_path / "extended_manifest.csv").exists():
    df_ext = pd.read_csv(extended_path / "extended_manifest.csv")
    print(f"Extended manifest: {len(df_ext):,} rows")
else:
    print("No extended manifest found. Run 08_extended_sources.ipynb first.")
    df_ext = pd.DataFrame()

In [ ]:
SYNONYMS = {
    "Xenochrophis piscator": "Fowlea piscator",
    "Xenochrophis vittatus": "Fowlea vittatus",
    "Trimeresurus malabaricus": "Craspedocephalus malabaricus",
    "Trimeresurus gramineus": "Craspedocephalus gramineus",
    "Trimeresurus trigonocephalus": "Craspedocephalus trigonocephalus",
    "Calloselasma rhodostoma": "Calloselasma rhodostoma",
    "Naja tripudians": "Naja naja",
}

OUR_SPECIES = set(classes_df["species"].tolist())

if len(df_ext) > 0:
    df_ext["species"] = df_ext["species"].map(lambda s: SYNONYMS.get(s, s))
    df_ext = df_ext[df_ext["species"].isin(OUR_SPECIES)]
    print(f"After taxonomy normalisation + species filter: {len(df_ext):,} rows")
    print("Species distribution in extended:")
    print(df_ext["species"].value_counts().head(20).to_string())

In [ ]:
if len(df_ext) > 0 and "bytes" in df_ext.columns:
    n0 = len(df_ext)
    df_ext = df_ext[df_ext["bytes"] >= 5000]
    print(f"After bytes filter: removed {n0 - len(df_ext)}, kept {len(df_ext)}")
if len(df_ext) > 0 and "width" in df_ext.columns and "height" in df_ext.columns:
    short_side = df_ext[["width","height"]].min(axis=1)
    n0 = len(df_ext)
    df_ext = df_ext[short_side >= 100]
    print(f"After size filter: removed {n0 - len(df_ext)}, kept {len(df_ext)}")

In [ ]:
# pHash deduplication: new photos must not match existing photos
# Load existing pHashes from processed dataset manifest
# For now, compute pHash from file path if images are available

def compute_phash(filepath, hash_size=8):
    try:
        img = Image.open(filepath).convert("L").resize((32, 32))
        return str(imagehash.phash(img, hash_size=hash_size))
    except:
        return None

# This step runs only if images are physically available
print("pHash dedup: run this step when image files are attached as input dataset.")
print("Logic:")
print("  1. Load existing pHashes from 03_process_dataset output")
print("  2. Compute pHash for each new photo")
print("  3. Drop if Hamming distance <= 4 to any existing pHash")
print("  4. Keep higher-resolution copy if duplicate within new batch")

In [ ]:
if len(df_ext) > 0:
    # Assign all new photos to train split
    df_ext["split"] = "train"
    df_ext["source"] = df_ext.get("source", "extended")
    
    # Combine
    df_combined = pd.concat([df_base, df_ext], ignore_index=True)
    
    # Leakage check: obs_id should not appear in both train and test
    test_obs = set(df_combined[df_combined["split"]=="test"]["obs_id"].dropna())
    train_obs = set(df_combined[df_combined["split"]=="train"]["obs_id"].dropna())
    overlap = test_obs & train_obs
    print(f"Leakage check (obs_id train vs test): {len(overlap)} overlaps (should be 0)")
    
    df_combined.to_csv("/kaggle/working/combined_splits.csv", index=False)
    print(f"Combined splits saved: {len(df_combined):,} rows")
    print(df_combined["split"].value_counts().to_string())
else:
    print("No extended data to combine. Base dataset unchanged.")
    df_base.to_csv("/kaggle/working/combined_splits.csv", index=False)

## Summary

In [ ]:
# Load saved combined CSV for final summary stats
out_path = Path("/kaggle/working/combined_splits.csv")
if out_path.exists():
    df_final = pd.read_csv(out_path)
    print("=== Combined Dataset Summary ===")
    print(f"Total rows:        {len(df_final):,}")
    print(f"Unique species:    {df_final['species'].nunique() if 'species' in df_final.columns else 'n/a'}")
    print(f"Split distribution:")
    print(df_final["split"].value_counts().to_string() if "split" in df_final.columns else "  n/a")
    if "source" in df_final.columns:
        print(f"Source distribution:")
        print(df_final["source"].value_counts().to_string())
else:
    print("combined_splits.csv not found — run previous cells.")